# 02 — Label Engineering

Derive the five multi-label fraud labels from raw Sparkov features.
**This notebook must be run before any modelling notebook.**

Labels derived here:
| Label | Name | Gated on `is_fraud`? |
|---|---|---|
| `L_V` | Velocity burst | Yes |
| `L_G` | Geographic anomaly | No |
| `L_C` | Category anomaly | No |
| `L_R` | Ring membership | Yes |
| `L_T` | Temporal anomaly | No |

**Output:** `data/processed/train_labeled.parquet`, `data/processed/test_labeled.parquet`

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')

from src.labels import derive_labels, LABEL_COLS
from src.evaluation import label_cooccurrence_matrix, cross_border_stats

In [ ]:
train = pd.read_csv('../data/raw/fraudTrain.csv')
test  = pd.read_csv('../data/raw/fraudTest.csv')
print(f'Train: {len(train):,} | Test: {len(test):,}')

In [ ]:
# Derive labels — this is the only place labels are created
# All thresholds are fixed constants in src/labels.py
train_labeled = derive_labels(train)
test_labeled  = derive_labels(test)

In [ ]:
# Validate prevalence (each label should be >= 1% to be useful)
for col in LABEL_COLS:
    pct = train_labeled[col].mean() * 100
    flag = '⚠ LOW' if pct < 1.0 else 'OK'
    print(f'{col}: {pct:.2f}%  {flag}')

print('\nCross-border stats:')
print(cross_border_stats(train_labeled))

In [ ]:
# Label cardinality distribution
import matplotlib.pyplot as plt
train_labeled['label_cardinality'].value_counts().sort_index().plot(
    kind='bar', title='Label cardinality distribution (train)'
)
plt.xlabel('|L| (number of active labels)')
plt.tight_layout()

In [ ]:
# Label co-occurrence matrix
import seaborn as sns
y = train_labeled[LABEL_COLS].values
cooc = label_cooccurrence_matrix(y, LABEL_COLS)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cooc, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax)
ax.set_title('Label co-occurrence rates (train)')
plt.tight_layout()

In [ ]:
# Save to processed
train_labeled.to_parquet('../data/processed/train_labeled.parquet', index=False)
test_labeled.to_parquet('../data/processed/test_labeled.parquet', index=False)
print('Saved.')